# TileDB-SOMA storage companion

Throwaway notebook for getting a closer feel for the structures described in `../../wiki/concepts/tiledb-soma-storage.md`.

The goal is not analysis. The goal is to inspect the layers:

1. Census collection
2. SOMA experiment
3. `obs` and `var` dataframes
4. RNA `X` sparse arrays
5. TileDB schema / fragment directories on S3

Most cells are metadata-only. The default live sample is intentionally tiny: up to 10 tissues and 10 `obs` rows per tissue. The only expression-matrix read is guarded by `RUN_LIVE_X_QUERY = False` because even tiny-looking gene filters can cause large cell-major fragment reads.

## 0. Imports and knobs

Use this from the repo environment. If imports fail, install through `uv add`, not `pip install`.

In [1]:
import importlib.metadata as md
import json
import textwrap
import time
from pathlib import Path

import cellxgene_census
import pandas as pd
import tiledb
import tiledbsoma

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

ORGANISM = "homo_sapiens"
MEASUREMENT = "RNA"
X_LAYER = "raw"
GENES = ["ADORA1", "ADORA2A", "ADORA2B", "ADORA3"]

# Tiny metadata sample. This is meant to be quick and disposable.
RUN_TINY_OBS_SAMPLE = True
N_TISSUES = 10
N_CELLS_PER_TISSUE = 10
OBS_COORD_WINDOW = 50_000

# Keep expression reads off while skimming. Flip only when you intentionally want a live X query.
RUN_LIVE_X_QUERY = False


## 1. Open the Census handle

`open_soma()` opens a lazy handle. This should not download the expression matrix. The context mirrors the fetch script's generous S3 timeouts so structure inspection is less likely to die on a slow network.

In [2]:
census = cellxgene_census.open_soma()
census

The "stable" release is currently 2025-11-08. Specify 'census_version="2025-11-08"' in future calls to open_soma() to ensure data consistency.


<Collection 's3://cellxgene-census-public-us-west-2/cell-census/2025-11-08/soma/' (open for 'r') (3 items)
    'census_data': 's3://cellxgene-census-public-us-west-2/cell-census/2025-11-08/soma/census_data' (unopened)
    'census_info': 's3://cellxgene-census-public-us-west-2/cell-census/2025-11-08/soma/census_info' (unopened)
    'census_spatial_sequencing': 's3://cellxgene-census-public-us-west-2/cell-census/2025-11-08/soma/census_spatial_sequencing' (unopened)>

## 2. Census top level

This is the first wiki layer: a remote collection with metadata in `census_info` and per-organism experiments in `census_data`.

In [3]:
def describe_soma_node(name, obj):
    uri = getattr(obj, "uri", None)
    print(f"{name}: {type(obj).__name__}")
    if uri:
        print(f"  uri: {uri}")
    if hasattr(obj, "keys"):
        print(f"  keys: {list(obj.keys())}")

describe_soma_node("census", census)
describe_soma_node("census_info", census["census_info"])
describe_soma_node("census_data", census["census_data"])

census: Collection
  uri: s3://cellxgene-census-public-us-west-2/cell-census/2025-11-08/soma/
  keys: ['census_data', 'census_info', 'census_spatial_sequencing']
census_info: Collection
  uri: s3://cellxgene-census-public-us-west-2/cell-census/2025-11-08/soma/census_info
  keys: ['datasets', 'organisms', 'summary', 'summary_cell_counts']
census_data: Collection
  uri: s3://cellxgene-census-public-us-west-2/cell-census/2025-11-08/soma/census_data
  keys: ['callithrix_jacchus', 'homo_sapiens', 'macaca_mulatta', 'mus_musculus', 'pan_troglodytes']


In [4]:
datasets = census['census_info']['datasets'].read()

In [5]:
d = datasets.concat().to_pandas()

In [6]:
d

,soma_joinid,citation,collection_id,collection_name,collection_doi,collection_doi_label,dataset_id,dataset_version_id,dataset_title,dataset_h5ad_path,dataset_total_cell_count
0,0,Publication: https://doi.org/10.1016/j.isci.2022.104097 Dataset Version: https://datasets.cellxgene.cziscience.com/6...,8e880741-bf9a-4c8e-9227-934204631d2a,High Resolution Slide-seqV2 Spatial Transcriptomics Enables Discovery of Disease-Specific Cell Neighborhoods and Pat...,10.1016/j.isci.2022.104097,Marshall et al. (2022) iScience,4eb29386-de81-452f-b3c0-e00844e8c7fd,66699060-0389-4fbd-b3a5-196b3b4e32d6,Spatial transcriptomics in mouse: Puck_191112_05,4eb29386-de81-452f-b3c0-e00844e8c7fd.h5ad,10888
1,1,Publication: https://doi.org/10.1016/j.isci.2022.104097 Dataset Version: https://datasets.cellxgene.cziscience.com/f...,8e880741-bf9a-4c8e-9227-934204631d2a,High Resolution Slide-seqV2 Spatial Transcriptomics Enables Discovery of Disease-Specific Cell Neighborhoods and Pat...,10.1016/j.isci.2022.104097,Marshall et al. (2022) iScience,78d59e4a-82eb-4a61-a1dc-da974d7ea54b,f64950a2-a3c8-490a-8431-7121eeb4f5f4,Spatial transcriptomics in mouse: Puck_191112_08,78d59e4a-82eb-4a61-a1dc-da974d7ea54b.h5ad,10250
2,2,Publication: https://doi.org/10.1016/j.isci.2022.104097 Dataset Version: https://datasets.cellxgene.cziscience.com/7...,8e880741-bf9a-4c8e-9227-934204631d2a,High Resolution Slide-seqV2 Spatial Transcriptomics Enables Discovery of Disease-Specific Cell Neighborhoods and Pat...,10.1016/j.isci.2022.104097,Marshall et al. (2022) iScience,add5eb84-5fc9-4f01-982e-a346dd42ee82,781a724a-b0f5-46c4-9a13-e6293ef4364f,Spatial transcriptomics in mouse: Puck_191109_20,add5eb84-5fc9-4f01-982e-a346dd42ee82.h5ad,12906
3,3,Publication: https://doi.org/10.1016/j.isci.2022.104097 Dataset Version: https://datasets.cellxgene.cziscience.com/9...,8e880741-bf9a-4c8e-9227-934204631d2a,High Resolution Slide-seqV2 Spatial Transcriptomics Enables Discovery of Disease-Specific Cell Neighborhoods and Pat...,10.1016/j.isci.2022.104097,Marshall et al. (2022) iScience,b020294c-ab82-4547-b5a7-63d8ffa575ed,96a79598-297b-4ade-a6c1-a431ab243548,Spatial transcriptomics in mouse: Puck_191112_13,b020294c-ab82-4547-b5a7-63d8ffa575ed.h5ad,15161
4,4,Publication: https://doi.org/10.1038/s41591-024-03215-z Dataset Version: https://datasets.cellxgene.cziscience.com/a...,a96133de-e951-4e2d-ace6-59db8b3bfb1d,HTAN/HTAPP Broad - Spatio-molecular dissection of the breast cancer metastatic microenvironment,10.1038/s41591-024-03215-z,Klughammer et al. (2024) Nat Med,d7476ae2-e320-4703-8304-da5c42627e71,ac9fe945-6784-48fa-a2d5-a8646196d37e,HTAPP-330-SMP-1082 scRNA-seq,d7476ae2-e320-4703-8304-da5c42627e71.h5ad,565
...,...,...,...,...,...,...,...,...,...,...,...
1840,1840,Publication: https://doi.org/10.1038/s41586-024-07069-w Dataset Version: https://datasets.cellxgene.cziscience.com/c...,45d5d2c3-bc28-4814-aed6-0bb6f0e11c82,"A single-cell transcriptional timelapse of mouse embryonic development, from gastrula to pup",10.1038/s41586-024-07069-w,Qiu et al. (2024) Nature,dcfa2614-7ca7-4d82-814c-350626eccb26,ca20ef35-13c0-4850-bec4-ba00e6bbd6f9,Major cell cluster: Mesoderm,dcfa2614-7ca7-4d82-814c-350626eccb26.h5ad,3267338
1841,1841,Publication: https://doi.org/10.1126/science.abl4896 Dataset Version: https://datasets.cellxgene.cziscience.com/5a49...,e5f58829-1a66-40b5-a624-9046778e74f5,Tabula Sapiens,10.1126/science.abl4896,The Tabula Sapiens Consortium* et al. (2022) Science,53d208b0-2cfd-4366-9866-c3c6114081bc,5a495302-b7cd-4bf9-853e-95627b00bb03,Tabula Sapiens - All Cells,53d208b0-2cfd-4366-9866-c3c6114081bc.h5ad,1136218
1842,1842,Publication: https://doi.org/10.1038/s41586-024-07069-w Dataset Version: https://datasets.cellxgene.cziscience.com/a...,45d5d2c3-bc28-4814-aed6-0bb6f0e11c82,"A single-cell transcriptional timelapse of mouse embryonic development, from gastrula to pup",10.1038/s41586-024-07069-w,Qiu et al. (2024) Nature,dcfd4feb-18a3-4b30-81d7-1b0c544a8ab3,a5a85963-8004-41a1-8eb5-ca65266d89c3,Wh

In [7]:
summary = census["census_info"]["summary"].read().concat().to_pandas()
summary

,soma_joinid,label,value
0,0,census_schema_version,2.4.0
1,1,census_build_date,2025-11-08
2,2,dataset_schema_version,7.0.0
3,3,total_cell_count,217768036
4,4,unique_cell_count,125463259


## 3. SOMA experiment layout

For an organism, SOMA gives us an `Experiment`: cell metadata (`obs`) plus one or more measurements. In this project we care about the RNA measurement.

In [14]:
ORGANISM

'homo_sapiens'

In [15]:
MEASUREMENT

'RNA'

In [11]:
human = census["census_data"][ORGANISM]
rna = human.ms[MEASUREMENT]

describe_soma_node("human experiment", human)
describe_soma_node("human.obs", human.obs)
describe_soma_node("human.ms", human.ms)
describe_soma_node("human.ms['RNA']", rna)
describe_soma_node("human.ms['RNA'].var", rna.var)
describe_soma_node("human.ms['RNA'].X", rna.X)

human experiment: Experiment
  uri: s3://cellxgene-census-public-us-west-2/cell-census/2025-11-08/soma/census_data/homo_sapiens
  keys: ['ms', 'obs']
human.obs: DataFrame
  uri: s3://cellxgene-census-public-us-west-2/cell-census/2025-11-08/soma/census_data/homo_sapiens/obs
  keys: ['soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars']
human.ms: Collection
  uri: s3://cellxgene-census-public-us-west-2/cell-census/2025-11-08/soma/census_data/homo_sapiens/ms
  keys: ['RNA']
human.ms['RNA']: Meas

## 4. Axis schemas: `obs` and `var`

These are metadata tables. Reading `var` for four genes is cheap. The `obs` sample below is deliberately tiny: 10 tissues x 10 cells, metadata only.

In [12]:
print("obs schema")
print(human.obs.schema)
print("\nvar schema")
print(rna.var.schema)

obs schema
soma_joinid: int64 not null
dataset_id: dictionary<values=large_string, indices=int16, ordered=0>
assay: dictionary<values=large_string, indices=int8, ordered=0>
assay_ontology_term_id: dictionary<values=large_string, indices=int8, ordered=0>
cell_type: dictionary<values=large_string, indices=int16, ordered=0>
cell_type_ontology_term_id: dictionary<values=large_string, indices=int16, ordered=0>
development_stage: dictionary<values=large_string, indices=int16, ordered=0>
development_stage_ontology_term_id: dictionary<values=large_string, indices=int16, ordered=0>
disease: dictionary<values=large_string, indices=int16, ordered=0>
disease_ontology_term_id: dictionary<values=large_string, indices=int16, ordered=0>
donor_id: dictionary<values=large_string, indices=int16, ordered=0>
is_primary_data: bool
observation_joinid: large_string
self_reported_ethnicity: dictionary<values=large_string, indices=int8, ordered=0>
self_reported_ethnicity_ontology_term_id: dictionary<values=larg

In [16]:
var_filter = "feature_name in " + repr(GENES)
adora_var = (
    rna.var.read(
        value_filter=var_filter,
        column_names=["soma_joinid", "feature_id", "feature_name", "feature_length"],
    )
    .concat()
    .to_pandas()
    .sort_values("feature_name")
)
adora_var

,soma_joinid,feature_id,feature_name,feature_length
2,15755,ENSG00000163485,ADORA1,2219
1,14173,ENSG00000128271,ADORA2A,691
3,16407,ENSG00000170425,ADORA2B,1698
0,658,ENSG00000282608,ADORA3,1067


In [17]:
RUN_TINY_OBS_SAMPLE

True

In [18]:
OBS_SAMPLE_COLUMNS = [
    "soma_joinid",
    "dataset_id",
    "cell_type",
    "tissue",
    "tissue_general",
    "disease",
    "is_primary_data",
]

if RUN_TINY_OBS_SAMPLE:
    obs_window = (
        human.obs.read(
            coords=(slice(0, OBS_COORD_WINDOW),),
            column_names=OBS_SAMPLE_COLUMNS,
        )
        .concat()
        .to_pandas()
    )
    tissue_order = (
        obs_window["tissue_general"]
        .astype("object")
        .value_counts()
        .head(N_TISSUES)
        .index.tolist()
    )
    obs_probe = (
        obs_window[obs_window["tissue_general"].astype("object").isin(tissue_order)]
        .assign(sample_tissue=lambda df: df["tissue_general"].astype("object"))
        .groupby("sample_tissue", sort=False, observed=True)
        .head(N_CELLS_PER_TISSUE)
        .reset_index(drop=True)
    )
    print(
        f"sampled {len(obs_probe):,} obs rows across "
        f"{obs_probe['sample_tissue'].nunique()} tissues "
        f"from the first {OBS_COORD_WINDOW:,} obs coordinates"
    )
    display(obs_probe)
else:
    obs_probe = pd.DataFrame(columns=["sample_tissue", *OBS_SAMPLE_COLUMNS])
    print("Skipped. Set RUN_TINY_OBS_SAMPLE = True in the first cell to sample obs metadata.")

sampled 100 obs rows across 10 tissues from the first 50,000 obs coordinates


,soma_joinid,dataset_id,cell_type,tissue,tissue_general,disease,is_primary_data,sample_tissue
0,0,d7476ae2-e320-4703-8304-da5c42627e71,endothelial cell,liver,liver,breast cancer,False,liver
1,1,d7476ae2-e320-4703-8304-da5c42627e71,malignant cell,liver,liver,breast cancer,False,liver
2,2,d7476ae2-e320-4703-8304-da5c42627e71,fibroblast,liver,liver,breast cancer,False,liver
3,3,d7476ae2-e320-4703-8304-da5c42627e71,fibroblast,liver,liver,breast cancer,False,liver
4,4,d7476ae2-e320-4703-8304-da5c42627e71,macrophage,liver,liver,breast cancer,False,liver
...,...,...,...,...,...,...,...,...
95,49545,eec804b9-2ae5-44f0-a1b5-d721e21257de,myeloid cell,respiratory airway,respiratory system,COVID-19,True,respiratory system
96,49546,eec804b9-2ae5-44f0-a1b5-d721e21257de,myeloid cell,respiratory airway,respiratory system,COVID-19,True,respiratory system
97,49547,eec804b9-2ae5-44f0-a1b5-d721e21257de,myeloid cell,respiratory airway,respiratory system,COVID-19,True,respiratory system
98,49548,eec804b9-2ae5-44f0-a1b5-d721e21257de,myeloid cell,respiratory airway,respiratory system,COVID-19,True,respiratory system


The tiny `obs_probe` read is metadata only. It reads a small coordinate window from `obs`, then locally keeps up to 10 cells for each of up to 10 tissues. That makes the sample quick and avoids a server-side scan for each tissue.

In [19]:
if not obs_probe.empty:
    display(obs_probe.groupby("sample_tissue", observed=True).size().rename("sampled_cells").reset_index())
    display(obs_probe[["sample_tissue", "cell_type", "dataset_id"]].head(30))
else:
    print("No obs metadata loaded yet.")

,sample_tissue,sampled_cells
0,blood,10
1,bone marrow,10
2,brain,10
3,endocrine gland,10
4,eye,10
5,immune system,10
6,liver,10
7,lung,10
8,prostate gland,10
9,respiratory system,10


,sample_tissue,cell_type,dataset_id
0,liver,endothelial cell,d7476ae2-e320-4703-8304-da5c42627e71
1,liver,malignant cell,d7476ae2-e320-4703-8304-da5c42627e71
2,liver,fibroblast,d7476ae2-e320-4703-8304-da5c42627e71
3,liver,fibroblast,d7476ae2-e320-4703-8304-da5c42627e71
4,liver,macrophage,d7476ae2-e320-4703-8304-da5c42627e71
5,liver,endothelial cell,d7476ae2-e320-4703-8304-da5c42627e71
6,liver,fibroblast,d7476ae2-e320-4703-8304-da5c42627e71
7,liver,endothelial cell,d7476ae2-e320-4703-8304-da5c42627e71
8,liver,monocyte,d7476ae2-e320-4703-8304-da5c42627e71
9,liver,endothelial cell,d7476ae2-e320-4703-8304-da5c42627e71


## 5. The X layer as a TileDB sparse array

`X['raw']` is the sparse cell x gene matrix. At the SOMA level, it is triples like `(cell soma_joinid, gene soma_joinid, value)`. At the TileDB level, it is an array directory with schema, commits, metadata, and fragments.

In [20]:
x_raw = rna.X[X_LAYER]
describe_soma_node(f"X[{X_LAYER!r}]", x_raw)
print("\nSOMA schema")
print(x_raw.schema)

X['raw']: SparseNDArray
  uri: s3://cellxgene-census-public-us-west-2/cell-census/2025-11-08/soma/census_data/homo_sapiens/ms/RNA/X/raw

SOMA schema
soma_dim_0: int64 not null
soma_dim_1: int64 not null
soma_data: float not null


In [21]:
def vfs_ls(uri, n=20):
    vfs = tiledb.VFS()
    entries = list(vfs.ls(uri))
    print(f"{uri}")
    print(f"{len(entries):,} entries")
    for entry in entries[:n]:
        print(" ", entry)
    if len(entries) > n:
        print(f"  ... {len(entries) - n:,} more")
    return entries

x_entries = vfs_ls(x_raw.uri)

TileDBError: S3: Error while listing with prefix 's3://cellxgene-census-public-us-west-2/cell-census/2025-11-08/soma/census_data/homo_sapiens/ms/RNA/X/raw/' and delimiter '/'[Error Type: 100] [HTTP Response Code: 301] [Exception: PermanentRedirect] [Remote IP: 52.217.119.50] [Request ID: Z7W1YGX25H9CJ187] [Headers: 'content-type' = 'application/xml' 'date' = 'Thu, 04 Jun 2026 07:35:18 GMT' 'server' = 'AmazonS3' 'transfer-encoding' = 'chunked' 'x-amz-bucket-region' = 'us-west-2' 'x-amz-id-2' = '4FWFswX/92pBXf8vwSLdenimzcMKrwJ+4SSOmDQVPwIFiRU/6ecYmtmqXi8n/iP4r8cJC9GR7FI=' 'x-amz-request-id' = 'Z7W1YGX25H9CJ187'] : Unable to parse ExceptionName: PermanentRedirect Message: The bucket you are attempting to access must be addressed using the specified endpoint. Please send all future requests to this endpoint.

In [ ]:
fragment_uri = x_raw.uri.rstrip("/") + "/__fragments"
commit_uri = x_raw.uri.rstrip("/") + "/__commits"

fragments = vfs_ls(fragment_uri, n=10)
commits = vfs_ls(commit_uri, n=10)

## 6. TileDB schema details

This is the closest view of the physical array: dimensions, attributes, tile extents, capacity, and filters. The key question from the wiki page is whether reads are naturally organized around the cell axis, the gene axis, or both.

In [ ]:
with tiledb.open(x_raw.uri, mode="r") as arr:
    schema = arr.schema
    print(schema)
    print("\nDomain dimensions")
    for dim in schema.domain:
        print(f"- {dim.name}: domain={dim.domain}, tile={dim.tile}, dtype={dim.dtype}")
    print("\nAttributes")
    for attr in schema:
        print(f"- {attr.name}: dtype={attr.dtype}, filters={attr.filters}")

## 7. Fragment directory peek

A fragment is the useful mental unit for S3 traffic. The exact file names can vary by TileDB version, but this gives a concrete sense of `__fragments/<fragment>/...` rather than treating it as abstract storage.

In [ ]:
if fragments:
    first_fragment = fragments[0]
    first_fragment_entries = vfs_ls(first_fragment, n=30)
else:
    print("No fragments listed.")

## 8. Why the four-gene query can still be expensive

The query below is intentionally disabled by default. It asks for four genes in one relatively narrow cell-type/tissue slice. The important lesson is that the returned `AnnData` can be tiny while the storage engine still had to inspect much larger cell-major fragments.

Flip `RUN_LIVE_X_QUERY` at the top only when you want to feel the timing.

In [ ]:
if RUN_LIVE_X_QUERY:
    obs_value_filter = (
        "tissue_general == 'brain' "
        "and cell_type == 'astrocyte' "
        "and disease == 'normal' "
        "and is_primary_data == True"
    )
    var_value_filter = "feature_name in " + repr(GENES)

    t0 = time.monotonic()
    adata = cellxgene_census.get_anndata(
        census=census,
        organism="Homo sapiens",
        measurement_name=MEASUREMENT,
        X_name=X_LAYER,
        obs_value_filter=obs_value_filter,
        var_value_filter=var_value_filter,
        obs_column_names=["cell_type", "tissue", "tissue_general", "dataset_id"],
    )
    dt = time.monotonic() - t0
    print(f"returned shape: {adata.shape}, nnz={adata.X.nnz:,}, seconds={dt:,.1f}")
    display(adata)
else:
    print("Skipped. Set RUN_LIVE_X_QUERY = True in the first cell to run this intentionally.")

## 9. Native iterator sketch

This is the direction hinted in the wiki page and fetch post-mortem: use axis queries and table iteration when we want checkpointable chunks rather than one monolithic `get_anndata()` call.

This cell is a sketch, not something to run casually.

In [ ]:
print(textwrap.dedent('''
with human.axis_query(
    measurement_name="RNA",
    obs_query=tiledbsoma.AxisQuery(value_filter="tissue_general == 'brain' and cell_type == 'astrocyte'"),
    var_query=tiledbsoma.AxisQuery(value_filter="feature_name in ['ADORA1', 'ADORA2A', 'ADORA2B', 'ADORA3']"),
) as query:
    for table in query.X("raw").tables():
        # table is a pyarrow Table of sparse X triples for one streamed chunk.
        # Write/checkpoint here instead of waiting for a giant AnnData.
        ...
'''))

## 10. Close handles

Run this when finished poking around.

In [ ]:
census.close()
print("closed")